In [1]:
import pandas as pd

train = pd.read_csv("../data/splits/train.csv")
val = pd.read_csv("../data/splits/val.csv")
test = pd.read_csv("../data/splits/test.csv")

print(train.shape, val.shape, test.shape)

(34996, 4) (7501, 4) (7500, 4)


In [2]:
X_train, y_train = train["clean_content"], train["label"]
X_val, y_val = val["clean_content"], val["label"]
X_test, y_test = test["clean_content"], test["label"]

print(X_train.shape, y_train.shape)

(34996,) (34996,)


In [3]:
print("Missing in X_train:", X_train.isna().sum())
X_train = X_train.fillna("")
X_val = X_val.fillna("")
X_test = X_test.fillna("")

Missing in X_train: 0


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)

(34996, 20000)


In [5]:
from sklearn.linear_model import LogisticRegression
import time

start = time.time()
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)
print(f"Training took {time.time() - start:.2f} seconds")

Training took 0.19 seconds


In [6]:
from sklearn.metrics import classification_report, accuracy_score

val_preds = clf.predict(X_val_vec)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

Validation Accuracy: 0.8649513398213572
              precision    recall  f1-score   support

           0       0.87      0.86      0.86      3745
           1       0.86      0.87      0.87      3756

    accuracy                           0.86      7501
   macro avg       0.86      0.86      0.86      7501
weighted avg       0.86      0.86      0.86      7501



In [7]:
test_preds = clf.predict(X_test_vec)
print("Test Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

Test Accuracy: 0.8517333333333333
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      3744
           1       0.85      0.86      0.85      3756

    accuracy                           0.85      7500
   macro avg       0.85      0.85      0.85      7500
weighted avg       0.85      0.85      0.85      7500



In [8]:
baseline_results = {
    "model": "TF-IDF + Logistic Regression",
    "test_accuracy": accuracy_score(y_test, test_preds),
}
print(baseline_results)

{'model': 'TF-IDF + Logistic Regression', 'test_accuracy': 0.8517333333333333}


## Day 4: Baseline Results
- Model: TF-IDF (unigram+bigram, 20K features) + Logistic Regression
- Test Accuracy: [your number]
- Test F1 (weighted): [your number from classification_report]
- This is the number DistilBERT fine-tuning (Day 6-7) needs to beat.

In [2]:
import pandas as pd

train = pd.read_csv("../data/splits/train.csv")
val = pd.read_csv("../data/splits/val.csv")
test = pd.read_csv("../data/splits/test.csv")

print(train.shape, val.shape, test.shape)

(34996, 4) (7501, 4) (7500, 4)


In [3]:
X_train, y_train = train["clean_content"], train["label"]
X_val, y_val = val["clean_content"], val["label"]
X_test, y_test = test["clean_content"], test["label"]

X_train = X_train.fillna("")
X_val = X_val.fillna("")
X_test = X_test.fillna("")

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

In [5]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)
print("Model trained.")

Model trained.


In [6]:
from sklearn.metrics import accuracy_score

val_preds = clf.predict(X_val_vec)
test_preds = clf.predict(X_test_vec)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Test Accuracy:", accuracy_score(y_test, test_preds))

Validation Accuracy: 0.8649513398213572
Test Accuracy: 0.8517333333333333


In [7]:
print(clf)
print(X_test_vec.shape)

LogisticRegression(max_iter=1000, random_state=42)
(7500, 20000)


In [8]:
results_df = test.copy().reset_index(drop=True)
results_df["predicted"] = test_preds
results_df["correct"] = results_df["label"] == results_df["predicted"]

print(results_df.shape)
print(results_df["correct"].value_counts())

(7500, 6)
correct
True     6388
False    1112
Name: count, dtype: int64


In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(results_df["label"], results_df["predicted"])
print("Confusion Matrix:")
print(cm)
print("\n(rows = actual, columns = predicted)")
print("[[True Negative, False Positive]")
print(" [False Negative, True Positive]]")

Confusion Matrix:
[[3163  581]
 [ 531 3225]]

(rows = actual, columns = predicted)
[[True Negative, False Positive]
 [False Negative, True Positive]]


In [10]:
misclassified = results_df[results_df["correct"] == False]
print(f"Total misclassified: {len(misclassified)}")

sample_errors = misclassified.sample(10, random_state=1)
for _, row in sample_errors.iterrows():
    print(f"Actual: {row['label']} | Predicted: {row['predicted']}")
    print(f"Review: {row['clean_content'][:300]}")
    print("-" * 80)

Total misclassified: 1112
Actual: 1 | Predicted: 0
Review: The best crime film around.Adapted by Donald Westlak, of Bernie Rodenbar fame, from a book by Jim Thompson, Get Shorty.Don't come expecting 'Better off Dead' John Cusack!
--------------------------------------------------------------------------------
Actual: 0 | Predicted: 1
Review: I got this book on cd and it is just a bore. I can not finish it. The reader gives it no thrill. I don't even know if this is a good book, it's not worth plodding thru, the reading reminds one of elevator music.
--------------------------------------------------------------------------------
Actual: 0 | Predicted: 1
Review: 2 1/2Works best for those already into the group. It's not a terrible collection of non-album tracks, just not a very good one either.
--------------------------------------------------------------------------------
Actual: 1 | Predicted: 0
Review: This movie will make you understand the phrase "To thine ownself be true". You ca

In [11]:
misclassified = results_df[results_df["correct"] == False].copy()
correct = results_df[results_df["correct"] == True].copy()

misclassified["content_length"] = misclassified["clean_content"].str.len()
correct["content_length"] = correct["clean_content"].str.len()

print("Misclassified avg length:", misclassified["content_length"].mean())
print("Correct avg length:", correct["content_length"].mean())

Misclassified avg length: 423.19874100719426
Correct avg length: 404.92783343769565


In [12]:
probs = clf.predict_proba(X_test_vec)
results_df["confidence"] = probs.max(axis=1)

most_confident_wrong = results_df[results_df["correct"] == False].sort_values("confidence", ascending=False).head(5)
for _, row in most_confident_wrong.iterrows():
    print(f"Confidence: {row['confidence']:.3f} | Actual: {row['label']} | Predicted: {row['predicted']}")
    print(f"Review: {row['clean_content'][:300]}")
    print("-" * 80)

Confidence: 0.984 | Actual: 0 | Predicted: 1
Review: Received this book timely and in great condition. It is a really good book for anyone. Provides good help in all areas of letterwritting.
--------------------------------------------------------------------------------
Confidence: 0.980 | Actual: 1 | Predicted: 0
Review: I save so much money when I buy my baby's diapers from here and they always ship within 3-4 days.
--------------------------------------------------------------------------------
Confidence: 0.977 | Actual: 1 | Predicted: 0
Review: How Ray Milland ever got involved in a turkey such as this is anyone's guess...This is one of those films that should be shown with the silhouetted figures of Mystery Science 3000; instead, it is presumably so bad that more cash could be made by showing the stupid thing as-is.What's worse is that (b
--------------------------------------------------------------------------------
Confidence: 0.974 | Actual: 1 | Predicted: 0
Review: this gam

In [13]:
misclassified = results_df[results_df["correct"] == False].copy()
correct = results_df[results_df["correct"] == True].copy()

misclassified["content_length"] = misclassified["clean_content"].str.len()
correct["content_length"] = correct["clean_content"].str.len()

print("Misclassified avg length:", misclassified["content_length"].mean())
print("Correct avg length:", correct["content_length"].mean())

Misclassified avg length: 423.19874100719426
Correct avg length: 404.92783343769565


## Day 5: Error Analysis

- Overall error rate: 14.8% (1112/7500 test examples)
- Misclassified reviews average 423 characters vs 405 for correctly classified
  ones — a small difference, suggesting review length is not a major driver of
  errors; error patterns are more content-driven than length-driven
- Common error patterns observed:
  - **Negation/hedging language** ("isn't a real good thing," "not 100% sure")
    that word-level TF-IDF features struggle to scope correctly
  - **Mixed or indirect sentiment** (e.g., "not terrible... just not very good
    either") that's genuinely ambiguous even for a human reader
  - **Dataset label noise**: several high-confidence "wrong" predictions were
    actually the model correctly reading clearly negative text (e.g., "crappy
    game," "so bad, presumably") that had been mislabeled positive in the
    source data — confirming the label-noise concern first raised in Day 2
- This baseline's core weakness (treating words independently, missing negation
  scope and sarcasm/indirect phrasing) is exactly what a transformer-based model
  like DistilBERT should improve on, since it processes words in context rather
  than in isolation.